# 01 — Findex audit

Load the Global Findex microdata, filter to the 20-country pool, check it matches the 40% mobile-money rule, and find columns that can't be used.

In [1]:
import pandas as pd

from mkopoguard import config

pd.set_option("display.max_columns", 50)
print(config.FINDEX_FILE)
print(config.FINDEX_FILE.exists())

D:\MkopoGuard\data\raw\micro_world_139countries.csv
True


In [2]:
raw = pd.read_csv(config.FINDEX_FILE, encoding=config.FINDEX_ENCODING, low_memory=False)
print(raw.shape)
raw.head(3)

(143887, 128)


,economy,economycode,regionwb,pop_adult,wpid_random,wgt,female,age,educ,inc_q,emp_in,urbanicity_f2f,account,account_fin,account_mob,fin1_1a,fin1_1b,fin2,fin4,fin4a,fin5,fin6,fin7,fin8,fin8a,...,fin42a,fin43a,fin43b,fin43d,fin43e,fin44a,fin44b,fin44c,fin44d,fin45,fin45_1,fin45_1_China,saved,borrowed,receive_wages,receive_transfers,receive_pension,receive_agriculture,pay_utilities,remittances,mobileowner,internetaccess,anydigpayment,merchantpay_dig,year
0,Afghanistan,AFG,South Asia,22647496.0,144274031,0.716416,2,43.0,2,4,1.0,1.0,1,1,0.0,NaN,NaN,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,1,2,3,4,1.0,1.0,NaN,0,1,4,4,4,4.0,1,5.0,1,2,1,0.0,2021
1,Afghanistan,AFG,South Asia,22647496.0,180724554,0.497408,2,55.0,1,3,1.0,1.0,0,0,0.0,NaN,NaN,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1.0,2.0,2.0,1.0,NaN,2,1,1,1,3.0,1.0,NaN,0,1,4,4,4,2.0,4,5.0,1,2,0,0.0,2021
2,Afghanistan,AFG,South Asia,22647496.0,130686682,0.650431,1,15.0,1,2,2.0,1.0,0,0,0.0,NaN,NaN,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,2,1,1,1,4.0,1.0,NaN,0,1,4,4,4,4.0,4,3.0,2,2,0,0.0,2021


In [3]:
ssa = raw[raw["regionwb"] == config.POOL_REGION]
mm_rate = ssa.groupby("economycode")["account_mob"].mean()
rule_countries = sorted(mm_rate[mm_rate >= config.MOBILE_MONEY_THRESHOLD].index)

assert rule_countries == sorted(config.POOL_COUNTRIES), "Config list doesn't match the 40% rule"
print(f"{len(rule_countries)} countries meet the rule and match the config ✔")

20 countries meet the rule and match the config ✔


In [4]:
pool = raw[raw["economycode"].isin(config.POOL_COUNTRIES)].copy()

assert len(pool) == config.EXPECTED_POOL_ROWS
assert pool["economycode"].nunique() == 20
print(f"Pool: {len(pool):,} rows from {pool['economycode'].nunique()} countries ✔")

Pool: 20,060 rows from 20 countries ✔


In [5]:
overview = (
    pool.groupby(["economycode", "economy"])
    .agg(
        rows=("economy", "size"),
        survey_year=("year", "first"),
        mobile_money_rate=("account_mob", "mean"),
    )
    .sort_values("mobile_money_rate", ascending=False)
    .round(3)
)
overview

,,rows,survey_year,mobile_money_rate
economycode,economy,,,
KEN,Kenya,1000,2021,0.706
GHA,Ghana,1000,2021,0.678
GAB,Gabon,1020,2021,0.665
SWZ,Eswatini,1000,2022,0.634
UGA,Uganda,1000,2021,0.579
SEN,Senegal,1000,2021,0.546
NAM,Namibia,1000,2021,0.514
ZWE,Zimbabwe,1000,2021,0.513
LSO,Lesotho,1025,2022,0.499


In [6]:
missing_by_country = pool.isna().groupby(pool["economycode"]).mean()

empty_somewhere = missing_by_country.columns[(missing_by_country == 1).any()]
complete = missing_by_country.columns[missing_by_country.max() == 0]
partly_missing = missing_by_country.columns.difference(complete).difference(empty_somewhere)

print(f"Columns with no gaps at all:              {len(complete)}")
print(f"Columns partly missing:                   {len(partly_missing)}")
print(f"Columns completely empty in some country: {len(empty_somewhere)}")
print(list(empty_somewhere))

Columns with no gaps at all:              50
Columns partly missing:                   68
Columns completely empty in some country: 10
['urbanicity_f2f', 'fin10_1a', 'fin10_1b', 'fin10_1c', 'fin10_1d', 'fin10_1e', 'fin14_2_China', 'fin14c_2_China', 'fin31b1_China', 'fin45_1_China']


In [7]:
pool_clean = pool.drop(columns=empty_somewhere)
print(pool_clean.shape)

config.DATA_INTERIM.mkdir(parents=True, exist_ok=True)
out_path = config.DATA_INTERIM / "findex_pool.parquet"
pool_clean.to_parquet(out_path, index=False)
print(f"Saved to {out_path}")

(20060, 118)
Saved to D:\MkopoGuard\data\interim\findex_pool.parquet


## Findings

- Pool confirmed: 20,060 respondents across 20 countries, with more than 40% mobile-money rule.
- Three countries (Eswatini, Lesotho, Botswana) were surveyed in 2022; check in step 1.2 that kept questions were asked the same way.
- `urbanicity_f2f` is empty for some countries (likely those surveyed by phone), so urban/rural can't be a feature across the pool.
- 68 columns are partly missing. Most are follow-up questions only asked when the previous answer was "yes", so their gaps are expected, not errors. Step 1.2 decides which to keep.
- Saved `data/interim/findex_pool.parquet` (20,060 × 118).